In [1]:
from langchain_groq import ChatGroq

In [3]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile"
)
response = llm.invoke("king of the jungle")
print(response.content)

The phrase "King of the Jungle" is often associated with the lion, due to its majestic appearance, powerful roar, and dominant position in its ecosystem. Lions are known for their:

1. **Regal appearance**: With their shaggy manes and powerful physiques, lions exude a sense of majesty and authority.
2. **Fearless hunting**: Lions are skilled predators, using coordinated attacks to bring down prey much larger than themselves.
3. **Social hierarchy**: Lions live in prides, with a dominant male (and sometimes female) leading the group and protecting its members.
4. **Territorial dominance**: Lions fiercely defend their territory, marking boundaries with their roar, scent, and visual displays.

However, it's worth noting that the term "King of the Jungle" is a bit misleading, as lions do not typically inhabit dense jungles. They are more commonly found in savannas, grasslands, and open woodlands.

Other animals, like the tiger or the elephant, could also be considered "kings" of their resp

In [25]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/manager-software-engineering-itc/job/R-82446")
page_data = loader.load().pop().page_content
print(page_data)





















Manager, Software Engineering, ITC














































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previ

In [49]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        
        {page_data}
        ### INSTRUCTION:
        
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
       
        Only return the valid JSON.
        
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [51]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Manager, Software Engineering, ITC',
 'experience': 'Minimum 7 years of relevant full stack engineering experience',
 'skills': ['Full stack development',
  'Databricks',
  'Generative AI',
  'Agentic AI frameworks',
  'LLMOps',
  'MLOps',
  'DevOps',
  'CI/CD pipelines',
  'AI quality culture',
  'Security',
  'Governance',
  'Access-control'],
 'description': 'We are searching for a Full Stack Engineering Manager with advanced AI skills to lead and oversee a diverse team of engineers, including Leads, SWE I, SWE II, and SWE III. The ideal candidate will have hands-on experience across the SDLC lifecycle and a proven ability to architect, develop, and deploy scalable solutions using Databricks, Generative AI, and Agentic AI frameworks.'}

In [53]:
type(json_res)

dict

In [33]:
import pandas as pd

df = pd.read_csv("app/resource/my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [35]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [37]:
links = collection.query(query_texts=["Experience in Python", "Expertise in React Native"], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/react-native-portfolio'},
  {'links': 'https://example.com/react-portfolio'}]]

In [45]:
job

{'role': 'Manager, Software Engineering, ITC',
 'experience': 'Minimum 7 years of relevant full stack engineering experience',
 'skills': ['Databricks',
  'Generative AI',
  'Agentic AI frameworks',
  'Full stack development',
  'LLMOps',
  'MLOps',
  'DevOps',
  'CI/CD pipelines'],
 'description': 'We are searching for a Full Stack Engineering Manager with advanced AI skills to lead and oversee a diverse team of engineers, including Leads, SWE I, SWE II, and SWE III.'}

In [43]:
job = json_res
job['skills']

['Databricks',
 'Generative AI',
 'Agentic AI frameworks',
 'Full stack development',
 'LLMOps',
 'MLOps',
 'DevOps',
 'CI/CD pipelines']

In [47]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Full Stack Engineering Management Solutions for Your Team

Dear Hiring Manager,

I came across your job posting for a Manager, Software Engineering, ITC, and I'm excited to introduce AtliQ, an AI & Software Consulting company that can help you find the perfect candidate to lead your diverse team of engineers. With our expertise in facilitating seamless integration of business processes through automated tools, I believe we can empower your enterprise to achieve scalability, process optimization, cost reduction, and heightened overall efficiency.

Our team at AtliQ has extensive experience in full stack engineering, and we're well-versed in the latest technologies, including Databricks, Generative AI, Agentic AI frameworks, LLMOps, MLOps, DevOps, and CI/CD pipelines. We've successfully delivered tailored solutions to numerous enterprises, and I'd like to highlight a few relevant examples from our portfolio:

* Our machine learning expertise is showcased in our Python por